In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [2]:
import json
import numpy as np
from src.data_loader import load_raw_data
from src.preprocessing import clean_raw_data, chronological_split, fit_scaler, apply_scaler, inverse_transform_column
from src.feature_engineering import build_features
from src.sequence_builder import create_sequences_for_all_splits
from src.evaluation.metrics import evaluate_all
from src.config import MODEL_FEATURE_COLUMNS, TARGET_COL, DEFAULT_LOOKBACK, FORECAST_HORIZON

DEV_LOOKBACK = 72  # temporary, for fast iteration — NOT the final 168

df_raw = load_raw_data()
df_clean = clean_raw_data(df_raw)
df_features = build_features(df_clean)
splits = chronological_split(df_features)

scale_columns = MODEL_FEATURE_COLUMNS + [TARGET_COL]
scaler = fit_scaler(splits.train, scale_columns)

train_scaled = apply_scaler(splits.train, scaler, scale_columns)
val_scaled = apply_scaler(splits.val, scaler, scale_columns)
test_scaled = apply_scaler(splits.test, scaler, scale_columns)

sequences = create_sequences_for_all_splits(train_scaled, val_scaled, test_scaled, MODEL_FEATURE_COLUMNS)
X_train, y_train_scaled = sequences["train"]
X_val, y_val_scaled = sequences["val"]
X_test, y_test_scaled = sequences["test"]

print("Ready:", X_train.shape, X_val.shape, X_test.shape)

2026-09-11 15:34:13 | INFO     | src.data_loader | Loading raw dataset from C:\Users\Ram\OneDrive\Desktop\energy-demand-lstm\data\raw\continuous_dataset.csv
2026-09-11 15:34:14 | INFO     | src.data_loader | Loaded raw dataset: 48048 rows, 17 columns
2026-09-11 15:34:14 | INFO     | src.preprocessing | Cleaning complete: 48048 rows, 4 flagged as diagnostic extreme events (|z| > 4.0)
2026-09-11 15:34:15 | INFO     | src.feature_engineering | Added national weather averages: temp_avg, humidity_avg, precip_avg, wind_avg
2026-09-11 15:34:15 | INFO     | src.feature_engineering | Added calendar features and cyclical encodings
2026-09-11 15:34:15 | INFO     | src.feature_engineering | Added lag features: ['lag_1', 'lag_24', 'lag_48', 'lag_168']
2026-09-11 15:34:16 | INFO     | src.feature_engineering | Added rolling features: rolling_mean_24, rolling_std_24, rolling_mean_168
2026-09-11 15:34:16 | INFO     | src.feature_engineering | Feature engineering complete: 48048 rows before, 47880 rows

In [3]:
from src.config import UNIVARIATE_FEATURE_COLUMNS
from src.sequence_builder import create_sequences_for_all_splits

uni_sequences = create_sequences_for_all_splits(
    train_scaled, val_scaled, test_scaled,
    feature_columns=UNIVARIATE_FEATURE_COLUMNS,
)
X_train_uni, y_train_uni = uni_sequences["train"]
X_val_uni, y_val_uni = uni_sequences["val"]
X_test_uni, y_test_uni = uni_sequences["test"]

print("Univariate shapes:", X_train_uni.shape, X_val_uni.shape, X_test_uni.shape)
print("Univariate feature count:", len(UNIVARIATE_FEATURE_COLUMNS))

2026-09-11 15:34:18 | INFO     | src.sequence_builder | Built 33421 sequences: X=(33421, 72, 7), y=(33421, 24) (lookback=72, horizon=24, dropped 95 boundary rows)
2026-09-11 15:34:18 | INFO     | src.sequence_builder | Split 'train': X=(33421, 72, 7), y=(33421, 24)
2026-09-11 15:34:18 | INFO     | src.sequence_builder | Built 7087 sequences: X=(7087, 72, 7), y=(7087, 24) (lookback=72, horizon=24, dropped 95 boundary rows)
2026-09-11 15:34:18 | INFO     | src.sequence_builder | Split 'val': X=(7087, 72, 7), y=(7087, 24)
2026-09-11 15:34:18 | INFO     | src.sequence_builder | Built 7087 sequences: X=(7087, 72, 7), y=(7087, 24) (lookback=72, horizon=24, dropped 95 boundary rows)
2026-09-11 15:34:18 | INFO     | src.sequence_builder | Split 'test': X=(7087, 72, 7), y=(7087, 24)
Univariate shapes: (33421, 72, 7) (7087, 72, 7) (7087, 72, 7)
Univariate feature count: 7


In [4]:
import os
import torch
torch.set_num_threads(os.cpu_count())

from src.models.lstm import LSTMForecaster
from src.training.trainer import train_model, predict

uni_model = LSTMForecaster(num_features=X_train_uni.shape[2])
uni_history = train_model(uni_model, X_train_uni, y_train_uni, X_val_uni, y_val_uni, epochs=10)

print("Epochs trained:", uni_history["epochs_trained"])
print("Parameters:", uni_history["num_parameters"])
print("Training time (s):", round(uni_history["training_time_seconds"], 1))

2026-09-11 15:35:07 | INFO     | src.training.trainer | Training LSTMForecaster | 20248 parameters | device=cpu
2026-09-11 15:35:07 | INFO     | src.training.trainer |   batch 0/131
2026-09-11 15:35:38 | INFO     | src.training.trainer |   batch 100/131
2026-09-11 15:35:51 | INFO     | src.training.trainer | Epoch 1/10 | train_loss=0.058284 | val_loss=0.021911
2026-09-11 15:35:51 | INFO     | src.training.trainer |   batch 0/131
2026-09-11 15:36:21 | INFO     | src.training.trainer |   batch 100/131
2026-09-11 15:36:32 | INFO     | src.training.trainer | Epoch 2/10 | train_loss=0.018448 | val_loss=0.012770
2026-09-11 15:36:32 | INFO     | src.training.trainer |   batch 0/131
2026-09-11 15:37:00 | INFO     | src.training.trainer |   batch 100/131
2026-09-11 15:37:19 | INFO     | src.training.trainer | Epoch 3/10 | train_loss=0.008684 | val_loss=0.009289
2026-09-11 15:37:19 | INFO     | src.training.trainer |   batch 0/131
2026-09-11 15:37:47 | INFO     | src.training.trainer |   batch 1

In [5]:
from src.evaluation.metrics import evaluate_all
from src.preprocessing import inverse_transform_column
from src.config import TARGET_COL

y_pred_uni_scaled = predict(uni_model, X_test_uni)
y_true_uni = inverse_transform_column(y_test_uni, scaler, scale_columns, TARGET_COL)
y_pred_uni = inverse_transform_column(y_pred_uni_scaled, scaler, scale_columns, TARGET_COL)

univariate_metrics = evaluate_all(y_true_uni, y_pred_uni)
print("Univariate LSTM:", univariate_metrics)

Univariate LSTM: {'MAE': 79.75889631180218, 'RMSE': 106.02803623522703, 'MAPE': 6.864089441490391, 'sMAPE': 6.684010120125448, 'R2': 0.6702142374431005}


In [6]:
import json
from pathlib import Path
import pandas as pd

experiments_path = Path.cwd().parent / "experiments" / "model_comparison.json"
with open(experiments_path) as f:
    all_results = json.load(f)

comparison = pd.DataFrame({
    "Univariate LSTM": univariate_metrics,
    "Multivariate LSTM": all_results["LSTM"],
}).T[["MAE", "RMSE", "MAPE", "sMAPE", "R2"]]

print(comparison)

all_results["Univariate LSTM"] = {
    **univariate_metrics,
    "training_time_seconds": uni_history["training_time_seconds"],
    "num_parameters": uni_history["num_parameters"],
}
with open(experiments_path, "w") as f:
    json.dump(all_results, f, indent=2)

print("\nSaved Univariate LSTM to experiments/model_comparison.json")

                         MAE        RMSE      MAPE     sMAPE        R2
Univariate LSTM    79.758896  106.028036  6.864089  6.684010  0.670214
Multivariate LSTM  67.422624   93.767627  5.773968  5.654699  0.742073

Saved Univariate LSTM to experiments/model_comparison.json
